# Endometriosis single-cell RNA-seq analysis

This notebook develops an analysis of the processed 10x Genomics single-cell RNA-seq data associated with [Marečková *et al.*, *Nature Genetics* (2024)](https://www.nature.com/articles/s41588-024-01873-w).

The planned workflow is organized into four high-level sections:

1. **Read and organize the count matrices** (current section)
2. Quality control and normalization
3. Cell-type annotation
4. Endometriosis-focused exploration of a selected cell population

> This first section only constructs sample-level `AnnData` objects from the original filtered count matrices. It intentionally does not filter, normalize, log-transform, integrate, or annotate cells.

## 1. Read and organize the count matrices

### Input format and relationship to the paper

Each `.tar.gz` archive in `data/` represents one donor–library combination and contains a Cell Ranger-style filtered feature-barcode matrix:

- `matrix.mtx.gz`: sparse integer UMI counts, stored as genes × cells on disk
- `features.tsv.gz`: Ensembl gene IDs, gene symbols, and feature types
- `barcodes.tsv.gz`: 10x cell barcodes

The paper aligned reads to **GRCh38-2020-A** and generated filtered count matrices with **Cell Ranger 6.0.2**. `scanpy.read_10x_mtx` reads this format and returns an `AnnData` object oriented as cells × genes. We use Ensembl IDs as the gene index because they are stable and unique, while retaining gene symbols in `adata.var`.

The archive name follows `<library_id>_<donor_id>`. A donor can occur in multiple libraries, so both identifiers are retained separately. The raw 10x barcode is also retained, but the `AnnData` cell index is prefixed with the complete sample ID to prevent barcode collisions across samples.

### 1.1 Imports and project paths

The path-discovery code allows the notebook to be launched either from the project root or from the `notebooks/` directory. This notebook uses the project Conda environment defined in `environment.yml`. In JupyterLab or another IDE, select the **Endometriosis study** kernel before running the notebook.

In [ ]:
from pathlib import Path
import shutil
import tarfile
import tempfile

import anndata as ad
import pandas as pd
import scanpy as sc
from scipy import sparse

sc.settings.verbosity = 2


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing the project's data folder."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a parent directory containing data/.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

### 1.2 Discover the sample archives

We enumerate archives rather than relying on a manually maintained sample list. Sorting the paths makes sample order reproducible. The table below provides a quick check of the discovered sample IDs and compressed file sizes before loading any matrices.

In [ ]:
archive_paths = sorted(DATA_DIR.glob("*.tar.gz"))

if not archive_paths:
    raise FileNotFoundError(f"No .tar.gz sample archives were found in {DATA_DIR}")

archive_manifest = pd.DataFrame(
    {
        "sample_id": [path.name.removesuffix(".tar.gz") for path in archive_paths],
        "archive": [path.name for path in archive_paths],
        "size_mb": [path.stat().st_size / 1024**2 for path in archive_paths],
    }
)

print(f"Discovered {len(archive_paths)} sample archives.")
display(archive_manifest)

### 1.3 Define a loader for one sample

`scanpy.read_10x_mtx` expects the three matrix files to be in a directory. The loader therefore copies only those required files from an archive into a temporary directory, reads them, and removes the temporary files automatically. The source archives remain unchanged.

For each sample, the loader:

1. validates that all three expected 10x files are present;
2. reads only `Gene Expression` features into a sparse `AnnData`;
3. records the original barcode, sample ID, library ID, and donor ID in `adata.obs`;
4. creates globally unique cell IDs of the form `<sample_id>:<barcode>`; and
5. records source information in `adata.uns`.

At this stage, `adata.X` remains the original unnormalized UMI count matrix.

In [ ]:
REQUIRED_10X_FILES = ("matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz")


def split_sample_id(sample_id: str) -> tuple[str, str]:
    """Split '<library_id>_<donor_id>' at its final underscore."""
    try:
        library_id, donor_id = sample_id.rsplit("_", maxsplit=1)
    except ValueError as error:
        raise ValueError(
            f"Sample ID {sample_id!r} does not match '<library_id>_<donor_id>'."
        ) from error
    return library_id, donor_id


def read_sample_archive(archive_path: Path) -> ad.AnnData:
    """Read one archived 10x matrix and attach sample provenance."""
    sample_id = archive_path.name.removesuffix(".tar.gz")
    library_id, donor_id = split_sample_id(sample_id)
    archive_prefix = f"work/{sample_id}"

    with tempfile.TemporaryDirectory(prefix=f"{sample_id}_") as temporary_dir:
        matrix_dir = Path(temporary_dir)

        with tarfile.open(archive_path, mode="r:gz") as archive:
            for filename in REQUIRED_10X_FILES:
                member_name = f"{archive_prefix}/{filename}"
                try:
                    member = archive.getmember(member_name)
                except KeyError as error:
                    raise FileNotFoundError(
                        f"{archive_path.name} is missing {member_name}."
                    ) from error

                source = archive.extractfile(member)
                if source is None:
                    raise OSError(f"Could not read {member_name} from {archive_path.name}.")

                with source, (matrix_dir / filename).open("wb") as destination:
                    shutil.copyfileobj(source, destination)

        adata = sc.read_10x_mtx(
            matrix_dir,
            var_names="gene_ids",
            make_unique=True,
            gex_only=True,
            cache=False,
        )

    # CSR is efficient for the later cell-wise QC calculations.
    if not sparse.isspmatrix_csr(adata.X):
        adata.X = sparse.csr_matrix(adata.X)

    original_barcodes = adata.obs_names.astype(str)
    adata.obs["barcode"] = original_barcodes
    adata.obs["sample_id"] = sample_id
    adata.obs["library_id"] = library_id
    adata.obs["donor_id"] = donor_id
    adata.obs_names = pd.Index(
        [f"{sample_id}:{barcode}" for barcode in original_barcodes],
        name="cell_id",
    )
    adata.var_names.name = "gene_id"

    if not adata.obs_names.is_unique:
        raise ValueError(f"Cell IDs are not unique within {sample_id}.")

    adata.uns["sample_id"] = sample_id
    adata.uns["library_id"] = library_id
    adata.uns["donor_id"] = donor_id
    adata.uns["source_archive"] = str(archive_path.relative_to(PROJECT_ROOT))
    adata.uns["matrix_description"] = (
        "Cell Ranger filtered, unnormalized gene-expression UMI counts"
    )

    return adata

### 1.4 Load all samples as `AnnData` objects

The objects are stored in a dictionary keyed by sample ID. Keeping them separate is useful for sample-level inspection and QC; concatenation will be handled deliberately in the next section. Because the matrices are sparse, zeros are not materialized in memory, but loading all samples can still require several gigabytes of RAM.

In [ ]:
sample_adatas: dict[str, ad.AnnData] = {}

for position, archive_path in enumerate(archive_paths, start=1):
    sample_id = archive_path.name.removesuffix(".tar.gz")
    print(f"[{position:>2}/{len(archive_paths)}] Reading {sample_id}")
    sample_adatas[sample_id] = read_sample_archive(archive_path)

print(f"\nLoaded {len(sample_adatas)} AnnData objects.")

### 1.5 Summarize the loaded objects

This manifest verifies the dimensions and sparse-matrix representation of every object. `n_cells` is the number of filtered barcodes assigned to that donor–library combination; it is not the number of independent biological replicates.

In [ ]:
sample_summary = pd.DataFrame(
    [
        {
            "sample_id": sample_id,
            "library_id": adata.uns["library_id"],
            "donor_id": adata.uns["donor_id"],
            "n_cells": adata.n_obs,
            "n_genes": adata.n_vars,
            "nonzero_values": adata.X.nnz,
            "matrix_dtype": str(adata.X.dtype),
            "sparse_format": adata.X.format,
        }
        for sample_id, adata in sample_adatas.items()
    ]
).sort_values("sample_id", ignore_index=True)

display(sample_summary)
print(f"Total matrix columns/cells: {sample_summary['n_cells'].sum():,}")
print(f"Unique donors: {sample_summary['donor_id'].nunique():,}")
print(f"Unique library IDs: {sample_summary['library_id'].nunique():,}")

### 1.6 Display the structure of one `AnnData` object

We select `UA_Endo10298211_FX1125` because it has already been inspected manually. Evaluating an `AnnData` object displays its dimensions and populated annotation slots. The following cells also preview its cell metadata (`obs`), gene metadata (`var`), sparse matrix properties, and stored provenance (`uns`).

In [ ]:
example_sample_id = "UA_Endo10298211_FX1125"
if example_sample_id not in sample_adatas:
    example_sample_id = next(iter(sample_adatas))

example_adata = sample_adatas[example_sample_id]
example_adata

In [ ]:
print("Cell annotations (adata.obs):")
display(example_adata.obs.head())

print("Gene annotations (adata.var):")
display(example_adata.var.head())

print("Count matrix (adata.X):")
print(f"  Python type: {type(example_adata.X).__name__}")
print(f"  Shape: {example_adata.X.shape} (cells × genes)")
print(f"  Data type: {example_adata.X.dtype}")
print(f"  Nonzero values: {example_adata.X.nnz:,}")

print("Stored provenance (adata.uns):")
display(example_adata.uns)

### Section 1 checkpoint

At the end of this section, `sample_adatas` contains one raw-count `AnnData` object per donor–library archive, and `sample_summary` describes the collection. No biological values have been changed.

The next section will add study metadata, concatenate the objects safely, calculate per-cell and per-gene QC metrics, and apply paper-aligned quality filters before normalization.